# Lesson 03 - エージェンシックデザインパターン

このレッスンでは、効果的なAIエージェントを構築するための3つの基本的なデザインパターンを探ります。

1. <strong>明確なエージェント指示</strong> — エージェントの動作を導く、役割を定義した正確なプロンプトの作成
2. **Pydanticモデルによる構造化出力** — エージェントが予測可能で検証済みのデータを返すことの保証
3. <strong>単一責任のエージェント</strong> — 各エージェントが一つのことをうまく行う、集中した設計

これらのパターンを、<strong>旅行先推薦システム</strong>のシナリオに適用し、目的地の提案から空き状況の確認、ロジスティクスの処理までを順に構築していきます。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## パターン1: エージェントへの明確な指示

最も効果的なパターンは、最もシンプルなものでもあります。エージェントに対して明確で詳細な指示を書くことです。

良い指示は以下を定義します：
- <strong>誰が</strong>エージェントであるか（ペルソナと口調）
- <strong>何を</strong>行うべきか（段階的な役割）
- <strong>どのように</strong>振る舞うべきか（制約やスタイル）

以下では、すべての応答に影響を与える明確な指示をもつ旅行コンシェルジュエージェントを作成します。


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## パターン2: Pydanticモデルによる構造化出力

自由形式のテキストは会話には便利ですが、下流のシステムでは構造化データが必要です。
<strong>Pydanticモデル</strong>と<strong>ツール関数</strong>を組み合わせることで、以下が可能になります。

- エージェントの出力に対する正確なスキーマを定義する
- 応答を自動的に検証する
- エージェントの結果をアプリケーションロジックに確実に統合する

また、目的地の詳細を返すツールを導入し、エージェントが実データに基づいて推奨を行えるようにします。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## パターン3: 単一責任エージェント

複雑なタスクは、1つの責任に集中した複数のエージェントに作業を分担させることで効果的になります:

- 場所や空き状況に詳しい <strong>目的地エキスパート</strong>
- フライト、ホテル、旅程を担当する <strong>物流プランナー</strong>

これはソフトウェア工学の原則である<em>関心の分離</em>を反映しています — 各エージェントは独立してテスト、保守、および改善が容易になります。


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## 概要

このレッスンでは、旅行推薦シナリオに対して3つのエージェント設計パターンを適用しました：

| パターン | 主要なアイデア | 利点 |
|---|---|---|
| <strong>明確な指示</strong> | ペルソナ、責任範囲、制約を事前に定義する | 一貫したブランドに沿ったエージェントの動作 |
| <strong>構造化された出力</strong> | Pydanticモデルを応答フォーマットとして使用する | 検証された機械可読な結果 |
| <strong>単一責任</strong> | 各エージェントに1つの集中した役割を与える | テスト、保守、組み合わせが容易 |

これらのパターンは自然に組み合わせられます。明確な指示と構造化された出力を単一責任のエージェント内で組み合わせることで、堅牢で実運用可能なシステムを構築できます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
